PROGETTO 6: ANALISI INTELLIGENTE DI CV IN PDF

Analisi automatizzate dei curriculum

Con questo si intende una pipeline che prende uno o più curriculum in formato PDF e cerca di trasformarli da documenti non strutturati a informazioni utilizzabili automaticamente
CV PDF -> estrazione del testo -> pulizia/normalizzazione -> NLP/AI -> estrazione delle informazioni -> dati strutturati -> analisi/confronto/classificazione

Quindi poi non lavori più con un PDF ma con dati strutturati.

Ci sono due tipi di PDF:
- PDF digitale: contiene vero testo ed è sufficiente una estrazione diretta. PDF -> PyMuPDF/pdfplumber -> testo
- PDF scansionato: è praticamente un'immagine che deve prima essere passata ad un OCR per poi estrarre il testo. PDF -> immagine -> EasyOCR/altro OCR -> testo

Fare OCR su un PDF che contiene già testo non è una buona idea perchè rischi di essere meno preciso. Esempio "SQL Server 2019" che diventa "SQL Server 2O19"

Una volta estratto il tessto devi capire la struttura del CV
Il testo estratto, per il computer, inizialmente è solo una sequenza di caratteri.
Bisogna capire che "Mario Rossi" -> persona "Milano" -> località "ABC srl" azienda -> "IT manager" ruolo, ecc
Qui entra in gioco NLP
Uno degli strumenti è il NER (Named Entity Recognition).
Che indivisua entità specifiche già decise (es. PERS, ORG, LOC)
Ma nei CV ci sono anche entità non standard che un NER generico potrebbe non riconoscere. Esempio "anni di esperienza", "certificazioni", ed altro.

In questi casi hai diverse soluzioni:
- Regex: per individuare una mail non serve necessariamente BERT, una regex può essere più affisabile, idem per i numeri di telefono e per alcune date.
- L'AI entra in gioco quando una semplice regex non basta, e qui che il Transformer o LLM diventano interessanti. Per esempio, dalla frase: "responsabile dell'implementazione e delle manutanzione dell'ERP aziendale, coordinamento fornitori e utenti interni" potresti dedurre ruole?ERP/IT Manager competenze: ERP, project management, IT manger
Qui una semplice regex non basta

Una volta che hai il CV in dati strutturati, puoi iniziare la vera analisi intelligente.
Per esempio potresti confrontare, automaticamente, le competenza necessarie alla posizione con le competenze del candidato, assegnado, ad ogni CV un punteggio. Rimanendo sempre valido che alla fine è l'human decision la cosa più importante.

Per trasformare un documento statico in un'analisi intelligente, dobbiamo seguire 3 atti fondamentali:

- Estrazione di testo strutturato tramite lbreria PyMuPDF
- Implementazione di un sistema NER personalizzato
- Sviluppo di un algoritmo di Ranking basato sulla similarità tra vettori di embedding.

Etrarre testo da un PDF è più difficle di quanto sembri

Persing del PDF: Oltre il Semplice Testo
Tecniche di estrazione dati da documenti non strutturati.
Hai ai provato a copiare ed incollare una tabella da un PDF? spesso il risultato è un disastro, questo perchè il pdf nasce per la visualizzazione non per la lettura dei dati.
I file PDF sono contenitori complessi progettati per la visualizzazione, non per il parsing dei dati. Estrarre testo mantenendo la coerenza sementica richiede strumenti in grado di interpretare la struttura della pagina senza perdere l'ordine logico delle informazioni.
Utilizzando PyMuPDF, noto anche come 'fitz', per la sua efficienza e capacità di accedere ai metadati dei blochi di testo, permettendoci di distinguere tra intestazioni, paragrafi e liste puntate tipiche dei curriculum vitae.

Per fare questo abbiamo bisogno di concetti chiari sulla struttura del documento

Concetti Chiave del Parsing PDF
Fondamentali per l'estrazione pulita del dato
- Document Object Model: comprensione della gerarchia tra pagine, blocchi di testo e linee individuali all'interno del file. pagine - blocchi di testo -linee
- Text Cleaning: ma non basta estrarre tuto, serve una pulizia profonda. Rimozione di caratteri speciali, stop-words e formattazione superflua che potrebbe disturbare i modelli di NLP. Rimoviamo tutto ciò che non aggiunge valore semantico.
- Encoding Management: gestione dei diversi set di caratteri per evitare errori di decodifica durante la trasformazione in stringhe Python
- Block Filtering: selezione dei solo blocchi rilevanti per escludere grafiche, icone o linee decorative presenti nel layout del CV

Lo strumento che ci permette di fare questo lavoro si chiama PyMuPDF

Funzionamento di PyMuPDF
E' incredibilmente veloce perchè scritto in C, ma facile da usare in Python.
A differenza di altri strumenti, ci permette di leggere i blocchi e questo è vitale nei pdf moderni, spesso si usa layout a due colonne, mentre un lettore base leggerebbe da sinistra a destra, il pymupdf rispetta il flusso dei blocchi.
- Apertura e Iterazione: il processo inizia caricando il file tramite fitz.open(). Iteriamo sulle pagine e, per ognuna, chiamiamo il metodo get_text con l'opzione blocks per ottenere coordinate e contenuto di ogni elemento.
- Ricostruzione del Flusso: PyMuPDF restituisce i blocchi seguendo l'ordine di lettura. Questo è cruciale per il CV, dove l'esperienza lavorativa è spesso elencata in ordine cronologico inverso o in colonne affiancate.
- Trattamento delle immagini: molti CV moderni includono grafici. Impareremo ad ignorare oggetti non testuali focalizzandoci esclusivamente sull'estrazione di metadati sementici utili al successivo step di Named Entity Recognition (NER)

Ma quanta fatica costa questo processo al computer?

Efficienza Computazionale del Parsing
Ottimizzazione dei tempi di lettura
L'efficienza di PyMPDF è legata alla sua implementazione in C.
Rispetto ad altre librerie, minimizza il consumo di RAM caricando solo le parti necessarie del documento.
Possiamo quantificare il throughput del sistema calcolando il tempo medio di estrazione per pagina in relazione alla densità dei blocchi presenti.
Il tempo è dato da: apertura file + estrazione dati + pulizia

Una volta che abbiamo il testo pulito, dobbiamo capire cosa c'è scritto dentro

NER per il Dominio Risorse Umane
Addestramento ed estrazione di entità specifiche.
Una volta ottenuto il testo grezzo, dobbiamo trasformarlo in dati strutturati.
Il NER (Named Entity Recognition) ci permette di 'etichettare' porzioni di testo come Nome del candidato, istruzione universitario, o competenze tecniche.
Dobbiamo passare da un blocco di testo informe ad un dizionario strutturato.
Utilizzeremo un approccio basato su modelli pre-addestrati come BERT, adattandoli tramite fine-tuning o utilizzando pipeline specifiche di SpaCy per identificare entità che non appartengono alle classi standard.
E' qui che il Deep Learning trasforma le parole in dati pronti per essere analizzati, magari da un algoritmo decisionale.

Quali sono gli ingrannaggi che muovono questo sistema di riconoscimento?

Componenti del Sistema NER
Elementi per la classificazione dei token
Il cuore del NER moderno è il contesto, modelli come BERT non guardano la sola parola, ma guardano cosa c'è interno. Usiamo la tokenizzazione per spezzare il testo e gli embedding contestuali per capire le parole. Inoltre personalizziamo questo sistema per le esigenze specifiche di un recruiter
- Tokenizzazione: scomposizione del testo del CV in unità minime elaborabili dal modello neurale.
- Labeling Schema: definizione di etichette personalizzate come SKILL, DEGREE e EXPERIENCE per la categorizzazione granulare
- Contextual Embeddings: utilizzo della poszione delle parole per distinguere tra un nome di persona e il nome dell'azienda.
- Entity Linking: connessione delle competenze estrata a un'ontologia o un database standarizzato di skill professionali.

Ingegneria delle Entità
- Estrazione del nome: il nome è solitamente l'entità più prominente all'inizio della pagina. Utilizziamo la posizione del blocco estratto da PyMuPDF come feature aggiuntiva per aumentare la confidenza del modello NER. In pratica usiamo le coordinate fornite da PyMuPDF come indizio  per un modello NER
- Identificazione skill: le competenze spesso appaiono in liste. Sfruttiao i modelli Transformer per riconoscere skill emergenti che non sono presenti in liste predefinite, basandoci sul contesto linguistico circostante.
- Validazione dell'Istruzione: rileviamo i titoli di studio cercando pattern che combinano il tipo di laua con il nome dell'istruzione, gestendo le variazioni linguistiche tra curricula diversi.

Come facciamo a sapere se il nostro investigatore digitale sta facendo un buon lavoro?

Valutazione del Modello NER
Metriche di accuratezza per entità
La performance di un sistema NER si misura tramite la precisione e il richiamo calcolati sui limiti dei token etichettati correttamente.
La metrica F1-score bilancia due componenti, garantendo che il modello non sia troppo aggressivo o troppo conservativo nell'identificazione delle skill.

Ora che abbiamo i dati, dobbiamo decidere chi è il candidato migliore.

Ranking Semantico dei Candidati
Confronto tra CV e Job Description tramite Embedding
L'ultimo passaggio consiste nel confrontare le informazioni estrastte con i requisiti di una posizione aperta. Invece di cercare parole chiave esatte, utilizzeremo la ricerca semantica per trovare i candidati più affini
Dimentica la vecchia ricerca per parole chiave. Se la jon description ricerca uno sviluppator esperto, un CV che dice senior software enginerring è un cv ottimo, anche se le parole sono diverse.
Entriamo nel regno della similarità semantica, dove confrontiamo concetti e non lettere.
Trasformeremo sia il CV che la Job Description in vettori in uno spazio latente ad alta dimensionalità La vicinanza tra questi due vettori indicherà quanto il profilo del candidato corrisponde alle necessità dell'azienda.

Applichiamo Algoritmo di Scoring
Metodi per il calcolo della rilevanza
- Sentence Embeddings: generazione di rappresentazioni vettoriali dense che catturano il significato globale delle esperienze lavorative
- Cosine Similarity: calcolo dell'angolo tra il vettore della Job Description e quello del CV per determinare il grado di affinità
- Weighted Skill Matching: assegnazione di pesi maggiori alle competenze 'must-have' definite nel profilo ricercato
- Rank Aggregation: combinazione dello score semantico con filtri boolenani basati su titoli di studio o anni di esperienza minimi.

Come calcoliamo matematicamente, questa vicinanza?

Implementazione del Ranking
- Generazione di Vettori: utilizziamo modelli come Sentence-BERT per codificare il riassunto delle esperienze. Questo permette di capire che 'sviluppatore Java' e 'Ingegnere del Software con esperienza in ambienti JVM' sono profili simili.
- Calcolo della Similarità: applichiamo la similarità coseno tra l'embedding della jon description e l'embedding del profilo estrastto. Un valore vicino a 1 indica una corrispondenza quasi perfettta
- Ordinamento dei Risultati: il sistema produce una lista decrescente di candidati, permettendo al recruiter di focalizzarsi solo sulla toop 5% dei profili, riducendo drasticamente il tempo di screening manuale.
Questo non sostiitsce l'umano, ma toglie il lavoro ripetivo iniziale, permettendoci di concentrarci sui colloqui e sulla parte empatica delle selezione.

Matematica della Similarità Coseno
Geometrica della rilevanza semantica
La similarità coseno misura il coseno dell'angolo tra due vettori diversi da zero, indicnado se puntano approssimativamente nella stessa direzione nello spazio delle feature.
Questra metrica è indipendente dalla lunghezza del testo (magnitudo del vettore), focalizzandosi esclusivamente sulla densità semantica del contenuto estratto.

CV / Testo candidato
-
Estrazione testo dal PDF
-
spaCy:  nome
        noun chunck
BERT:
        embbedding CV
        embedding Job Description
        dbdegging compotenze
- 
cosine similarity
- 
skill trovate + affinità CV/Job
-
score finale
-
classifica candidati


In [ ]:
#python -m spacy download en_core_web_sm
import os, re, fitz, torch, spacy
from transformers import AutoTokenizer, AutoModel
from pathlib import Path

# ==================================================================================
# PARAMETRI DI ANALISI SEMANTICA
# ==================================================================================
# Definiamo i concetti fondamentali che ricerchiamo nei candidati.
# Nota: Grazie al Deep Matching, il sistema troverà questi concetti anche se espressi 
# con parole diverse (es: "Coding" attiverà "Programming").

# PROFILO IDEALE
TARGET_CONCEPTS = ["python", "programming", "data science", "ai", "statistics", "mathematics", "sql"]

# Modello BERT: trasforma parole e frasi in vettori in uno spazio multidimensionale.
MODEL_NAME = "bert-base-uncased" #BERT inglese e uncased

# ==================================================================================
# CLASSE CORE: NeuralRecruiter (racchiude tutta la lofica)
# Un sistema di recruiting di nuova generazione che non si limita alle "parole chiave",
# ma comprende il significato dei concetti tecnici espressi nel CV.
# ==================================================================================
class NeuralRecruiter:
    def __init__(self):
        """Inizializza i componenti
         motori AI e indicizza i concetti target."""
        print("[SISTEMA] Avvio NeuralRecruiter...")
        
        # 1. Motore Linguistico (spaCy): Ottimo per estrarre strutture grammaticali (Nomi, Sostantivi).
        #spaCy utilizzato per la parte di estrazione informazioni di entità riconosciute 
        self.nlp = spacy.load("en_core_web_sm")  #anche SpaCy configurato in inglese come per BERT
        
        # 2. Motore Semantico (BERT): Il "cervello" che comprende i concetti.
        # per analisi semantica, per capire li testo
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) #testo in token
        self.model = AutoModel.from_pretrained(MODEL_NAME) #rappresentazioni vettoriali dei token
        
        # 3. Ottimizzazione (Pre-calcolo):
        # Calcoliamo i vettori dei concetti target una sola volta all'avvio (skill del profilo ideale). 
        # In questo modo, durante l'analisi dei CV, dovremo solo fare calcoli matematici rapidi.
        # per calcolare l'affinità con le skill del CV (quindi non cerco esattamente es. VB6 ma potrebbe andare bene anche Visual Basic 6)
        print("[SISTEMA] Indicizzazione vettoriale delle competenze target...")
        self.target_vecs = {t: self.get_emb(t) for t in TARGET_CONCEPTS}

    def get_emb(self, text): #funzione centrale per BERT
        """
        Trasforma qualsiasi testo (parola o frase) in un'impronta digitale numerica (vettore).
        Questo per poter utilizzare il vettore generato per il confronto.
        Utilizziamo la media degli stati dell'ultimo layer di BERT per una precisione massima.
        """
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
        with torch.no_grad(): #sto facendo solo inferenza, non devo addestrre BERT, non calcolare i gradienti/pesi
            output = self.model(**inputs)
            # Prendiamo la media di tutti i token per avere un vettore che rappresenti l'intero concetto
            return output.last_hidden_state.mean(dim=1)[0] #prende i vettori di tutti i token e ne calcola la media, ottengo un vettore unico

    def cosine_sim(self, v1, v2): #confronto di due vettori
        """Calcola quanto due vettori sono 'vicini' concettualmente (da 0.0 a 1.0).
        -1 = direzioni ooposte
         0  = ortogonali
        +1 = stessa direzione
        """
        
        return torch.nn.functional.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()

    def analyze(self, text, jd_vec):
        """
        Workflow di analisi multicriterio:
        Interazione: Combina estrazione linguistica (spaCy), Regex e modelli Transformer.
        """
        
        # --- 1. Estrazione Dati Strutturati ---
        # Email: usiamo Regex (affidabilità 100% su pattern fissi).
        # usiamo regex perchè una mail ha una struttura prevedibile quindi non è necessaria ai
        email = re.search(r'[\w\.-]+@[\w\.-]+', text)
        
        # Nome: usiamo spaCy NER per trovare entità di tipo 'PERSON'.
        # spaCy analizza tutto il CV ed estrae le entita PERSON, altrimenti mette SCONOSCIUTO
        doc = self.nlp(text)
        name = next((e.text for e in doc.ents if e.label_ == "PERSON" and "@" not in e.text), "Sconosciuto")

        # --- 2. Match Semantico Globale ---
        # Misuriamo l'affinità generale tra l'intero CV e la Job Description.
        cv_vec = self.get_emb(text)
        global_score = self.cosine_sim(cv_vec, jd_vec)

        # --- 3. Deep Skill Matching (Il punto di forza del sistema) ---
        # Non cerchiamo solo le parole esatte. Estraiamo tutti i "noun chunks" (concetti) dal CV.
        # QUI RITORNA IN GIOCO SPACY, che estrae gruppi nominali 
        cv_candidates = list(set([c.text.lower() for c in doc.noun_chunks if len(c.text) > 3]))
        found_skills = set()
        
        # Per ogni concetto trovato nel CV (tramite spacy), verifichiamo se è 'vicino' ai nostri Target Concepts.
        for word in cv_candidates:
            word_vec = self.get_emb(word) # Calcoliamo l'impronta del concetto rilevato
            for target, target_vec in self.target_vecs.items(): #confronto con le skill richieste
                # Se la somiglianza supera l'85%, consideriamo la skill come presente.
                # Esempio: "Neural Networks" è molto vicino a "AI", quindi attiverà il match.
                if self.cosine_sim(word_vec, target_vec) > 0.85: #skill presente
                    found_skills.add(target)
                    break 

        # --- 4. Calcolo Punteggio Finale Bilanciato ---
        # Pesiamo la similarità globale (70%) e aggiungiamo un bonus per le skill specifiche (30%).
        bonus = min(len(found_skills) * 0.05, 0.30)
        final_score = (global_score * 0.7) + bonus

        return {
            "name": name, 
            "email": email.group() if email else "N.D.",
            "score": final_score,
            "skills": list(found_skills)
        }

# ==================================================================================
# UTILITY DI GESTIONE INPUT
# ==================================================================================
def get_text_from_source(source):
    """Gestore polimorfico: accetta sia stringhe di testo che percorsi a file PDF."""
    if source.endswith(".pdf") and os.path.exists(source):
        with fitz.open(source) as doc:
            # Estraiamo il testo e puliamo i ritorni a capo per non confondere BERT
            return " ".join([p.get_text() for p in doc]).replace("\n", " ")
    return source

# ==================================================================================
# ESECUZIONE TEST E CLASSIFICA
# ==================================================================================
if __name__ == "__main__":
    # Inizializzazione dell'intelligenza artificiale
    bot = NeuralRecruiter()
    
    # 1. Recupera la cartella esatta dove si trova questo script (.py oppure .ipynb)
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__)) # per file .py
    except NameError:
        script_dir = os.getcwd()  # per file .ipynb

    # 2. Crea il percorso completo unendo la cartella dello script al nome del file
    pdf_path = os.path.join(script_dir, "cv_test.pdf")
    print(f"{pdf_path}")
    if not os.path.exists(pdf_path):
        print("[SETUP] Creazione CV di test in PDF...")
        doc = fitz.open()
        doc.new_page().insert_text((50,50), "Giulia Verdi. giulia@mail.com. Expert in Deep Learning and Calculus.")
        doc.save(pdf_path)

    # Obiettivo: Cosa stiamo cercando oggi?
    query_recruiting = "Looking for an expert in AI and Mathematics."
    jd_vec = bot.get_emb(query_recruiting)

    # Dataset di prova: Un mix di profili testuali e documenti PDF
    candidates = [
        "Mario Rossi. mario@mail.com. I love Social Media and Marketing.", # Profilo non in linea
        "Luca Bianchi. luca@tech.it. Experienced in Coding, Algorithms and Neural Networks.", # Ottimo (usa sinonimi)
        pdf_path # Profilo da PDF (Deep Learning + Calculus)
    ]

    results = []
    print(f"\n{'='*20} AVVIO ANALISI NEURALE {'='*20}")
    
    # Processiamo ogni candidato nel dataset
    for src in candidates:
        text = get_text_from_source(src)
        res = bot.analyze(text, jd_vec)
        results.append(res)
        print(f" -> Analizzato: {res['name']:<15} | Skill rilevate: {res['skills']}")

    # Ordinamento meritocratico basato sul punteggio finale
    results.sort(key=lambda x: x['score'], reverse=True)
    
    # Presentazione Tabellare dei Risultati
    print(f"\n{'POS':<5} {'FIT SCORE':<12} {'NOME':<20} {'CONCETTI RILEVATI (SEMANTICI)'}")
    print("-" * 85)
    for i, r in enumerate(results, 1):
        concept_str = ", ".join(r['skills']) if r['skills'] else "Nessuna affinità specifica"
        print(f"{i:<5} {r['score']*100:>6.1f}%      {r['name']:<20} {concept_str}")

[SISTEMA] Avvio NeuralRecruiter...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3745.87it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[SISTEMA] Indicizzazione vettoriale delle competenze target...
c:\EPICODE\Epicode_Python_AI_MachineLearning\PYTHON\Modulo_5_ComputerVisione_NPL\06_ProgettiFinali\cv_test.pdf
[SETUP] Creazione CV di test in PDF...

==================== AVVIO ANALISI NEURALE ====================
 -> Analizzato: Mario Rossi     | Skill rilevate: ['programming']
 -> Analizzato: Luca Bianchi    | Skill rilevate: ['ai']
 -> Analizzato: Giulia Verdi    | Skill rilevate: ['ai']

POS   FIT SCORE    NOME                 CONCETTI RILEVATI (SEMANTICI)
-------------------------------------------------------------------------------------
1       57.7%      Luca Bianchi         ai
2       52.7%      Giulia Verdi         ai
3       46.8%      Mario Rossi          programming
